[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/02_sumario_e_scd.ipynb)

# 02 - Processamento de Sumário e Atualização SCD Tipo II

## Configuração e Autenticação

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
from datetime import datetime

auth.authenticate_user()
project_id = 'projetotestemaua5584'  # Altere para o seu Project ID
client = bigquery.Client(project=project_id)

## 1. Criar Dataset TEMP
(Requisito 3)

In [ ]:
dataset_id = f"{project_id}.TEMP"
try:
    client.get_dataset(dataset_id)
    print("Dataset TEMP já existe.")
except Exception:
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"
    client.create_dataset(dataset)
    print("Dataset TEMP criado.")

## 2. Criar Tabela Resumo `dados_brasil_covid_censo`
(Requisito 4) - Utiliza SQL para combinar os dados mais recentes de COVID com o Censo.

In [ ]:
sql = f"""
CREATE OR REPLACE TABLE `{project_id}.dados_brasil.dados_brasil_covid_censo` AS
WITH latest_covid AS (
  SELECT * EXCEPT(row_num)
  FROM (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY ibgeID ORDER BY date DESC) as row_num
    FROM `{project_id}.dados_brasil.dados_brasil_covid`
  )
  WHERE row_num = 1
)
SELECT 
  c.date,
  c.state,
  c.city,
  c.ibgeID,
  c.totalCases,
  c.deaths,
  n.ESPVIDA,
  n.E_ANOSESTUDO,
  n.T_ANALF18M,
  n.RDPC,
  n.IDHM
FROM latest_covid c
JOIN `{project_id}.dados_brasil.dados_brasil_censo` n ON CAST(c.ibgeID AS STRING) = CAST(n.ibgeID AS STRING)
"""

query_job = client.query(sql)
query_job.result()
print("Tabela resumo criada com sucesso!")

## 3. Desafio Final: Atualização SCD Tipo II
(Requisito 6) - Processo para atualizar `dados_brasil_covid` utilizando conceitos de SCD Tipo II (Histórico de alterações).

In [ ]:
# Adicionar colunas de SCD Tipo II se não existirem
table_id = f"{project_id}.dados_brasil.dados_brasil_covid"

sql_alter = f"""
ALTER TABLE `{table_id}`
ADD COLUMN IF NOT EXISTS valid_from TIMESTAMP,
ADD COLUMN IF NOT EXISTS valid_to TIMESTAMP,
ADD COLUMN IF NOT EXISTS is_current BOOLEAN;
"""
client.query(sql_alter).result()

# Inicializar valores nulos se for a primeira vez que as colunas são adicionadas
sql_init = f"""
UPDATE `{table_id}`
SET valid_from = TIMESTAMP(date),
    valid_to = NULL,
    is_current = TRUE
WHERE valid_from IS NULL;
"""
client.query(sql_init).result()
print("Schema de SCD Tipo II preparado.")

In [ ]:
# 1. Obter dados novos (Simulação)
! wget --no-check-certificate --content-disposition 'https://github.com/wcota/covid19br/blob/master/cases-brazil-cities-time.csv.gz?raw=true' -O new_cases.csv.gz
! gunzip -f new_cases.csv.gz
df_new = pd.read_csv('new_cases.csv')
df_new = df_new[df_new['state'] != 'TOTAL']
df_new = df_new[['ibgeID', 'date', 'state', 'city', 'totalCases', 'deaths']]

# 2. Carregar para staging
staging_table = f"{project_id}.TEMP.staging_covid"
job = client.load_table_from_dataframe(df_new, staging_table, 
                                       job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job.result()

# 3. Executar Processo SCD Tipo II
merge_sql = f"""
-- 1. Desativar registros antigos que mudaram
UPDATE `{table_id}` T
SET valid_to = CURRENT_TIMESTAMP(), is_current = FALSE
WHERE is_current = TRUE 
  AND EXISTS (
    SELECT 1 FROM `{staging_table}` S 
    WHERE T.ibgeID = S.ibgeID AND T.date = S.date
      AND (T.totalCases != S.totalCases OR T.deaths != S.deaths)
  );

-- 2. Inserir novos registros ou novas versões
INSERT INTO `{table_id}` (ibgeID, date, state, city, totalCases, deaths, valid_from, valid_to, is_current)
SELECT 
  S.ibgeID, S.date, S.state, S.city, S.totalCases, S.deaths, 
  CURRENT_TIMESTAMP() as valid_from, NULL as valid_to, TRUE as is_current
FROM `{staging_table}` S
WHERE NOT EXISTS (
  SELECT 1 FROM `{table_id}` T 
  WHERE T.ibgeID = S.ibgeID AND T.date = S.date AND T.is_current = TRUE
);
"""

client.query(merge_sql).result()
print("Processo SCD Tipo II finalizado!")

## 4. Proposta de Visão de Dashboard
(Requisito 5.e)

**Visão Proposta:** Taxa de Letalidade vs. IDHM por UF.

- **Métrica:** (Total de Mortes / Total de Casos) * 100
- **Objetivo:** Identificar se estados com maior IDHM apresentam menor taxa de letalidade.
- **Gráfico:** Scatter Plot (Dispersão) no Power BI / Looker, onde o eixo X é o IDHM e o eixo Y é a Letalidade (%).